# Phase 21 — Ensemble: BERT-base Focal + DistilBERT + Multi-task BERT & BERT-base Focal + Multi-task only


**Μοντέλα:**
- BERT-base + Focal Loss, 20 epochs → Kaggle: 0.80399
- DistilBERT, 20 epochs → Kaggle: 0.76064
- Multi-task BERT + Focal Loss, 35 epochs → Kaggle: 0.77060

**Best ensemble so far:** 0.80548 (BERT-base 0.8 + DistilBERT 0.2)

**Μέθοδος:** Weighted average των probabilities

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

# BERT-base Focal (0.80399)
bertbase_hazard_probs  = np.load('npy/bertbase_focal_test_hazard_probs2.npy')
bertbase_product_probs = np.load('npy/bertbase_focal_test_product_probs2.npy')

# DistilBERT (0.76064)
distil_hazard_probs  = np.load('npy/bert_fulldata_test_hazard_probs.npy')
distil_product_probs = np.load('npy/bert_fulldata_test_product_probs.npy')

# Multi-task BERT 35 epochs (0.77060)
multitask_hazard_probs  = np.load('npy/multitask_test_hazard_probs.npy')
multitask_product_probs = np.load('npy/multitask_test_product_probs.npy')

print(f'BERT-base probs:   {bertbase_hazard_probs.shape}')
print(f'DistilBERT probs:  {distil_hazard_probs.shape}')
print(f'Multi-task probs:  {multitask_hazard_probs.shape}')

BERT-base probs:   (997, 10)
DistilBERT probs:  (997, 10)
Multi-task probs:  (997, 10)


In [2]:
# Δοκιμάζουμε συνδυασμούς βαρών
# Δίνουμε πάντα το μεγαλύτερο βάρος στο BERT-base (best single model)
#weight_combinations = [
    # (bert-base, distilbert, multitask)
    #(1.0, 0.0, 0.0),   # BERT-base μόνο (baseline)
    #(0.8, 0.2, 0.0),   # Best ensemble so far
    #(0.7, 0.2, 0.1),
    #(0.7, 0.1, 0.2),
    #(0.6, 0.2, 0.2),                             1st try
    #(0.6, 0.3, 0.1),
    #(0.6, 0.1, 0.3),
    #(0.5, 0.3, 0.2),
    #(0.5, 0.2, 0.3),
    #(0.5, 0.25, 0.25),
#]

#weight_combinations = [
    #(0.9, 0.0, 0.1),
    #(0.8, 0.0, 0.2),                           2nd try
    #(0.7, 0.0, 0.3),
    #(0.6, 0.0, 0.4),
    #(0.5, 0.0, 0.5),
#]


weight_combinations = [
    (0.1, 0.0, 0.9),
    (0.2, 0.0, 0.8),
    (0.3, 0.0, 0.7),
    (0.4, 0.0, 0.6),
    (0.5, 0.0, 0.5),
]

for w_bert, w_distil, w_multi in weight_combinations:
    hazard_avg  = (w_bert * bertbase_hazard_probs +
                   w_distil * distil_hazard_probs +
                   w_multi * multitask_hazard_probs)
    product_avg = (w_bert * bertbase_product_probs +
                   w_distil * distil_product_probs +
                   w_multi * multitask_product_probs)

    test_hazard  = le_hazard.inverse_transform(hazard_avg.argmax(axis=1))
    test_product = le_product.inverse_transform(product_avg.argmax(axis=1))

    filename = f'submission_b{int(w_bert*10)}_d{int(w_distil*10)}_m{int(w_multi*10)}.csv'
    pd.DataFrame({
        'id': test['id'],
        'hazard-category': test_hazard,
        'product-category': test_product
    }).to_csv(filename, index=False)
    print(f'BERT={w_bert:.1f}, Distil={w_distil:.1f}, Multi={w_multi:.1f} → {filename}')

print('1. submission_b7_d2_m1.csv  (0.7, 0.2, 0.1)')
print('2. submission_b7_d1_m2.csv  (0.7, 0.1, 0.2)')
print('3. submission_b6_d2_m2.csv  (0.6, 0.2, 0.2)')
print('4. submission_b6_d3_m1.csv  (0.6, 0.3, 0.1)')

BERT=0.1, Distil=0.0, Multi=0.9 → submission_b1_d0_m9.csv
BERT=0.2, Distil=0.0, Multi=0.8 → submission_b2_d0_m8.csv
BERT=0.3, Distil=0.0, Multi=0.7 → submission_b3_d0_m7.csv
BERT=0.4, Distil=0.0, Multi=0.6 → submission_b4_d0_m6.csv
BERT=0.5, Distil=0.0, Multi=0.5 → submission_b5_d0_m5.csv
1. submission_b7_d2_m1.csv  (0.7, 0.2, 0.1)
2. submission_b7_d1_m2.csv  (0.7, 0.1, 0.2)
3. submission_b6_d2_m2.csv  (0.6, 0.2, 0.2)
4. submission_b6_d3_m1.csv  (0.6, 0.3, 0.1)


# Distil doesnt offer an improvement anymore.
# For the 2nd try we only use bert and multitask and the score improves as we increase the weight of multitask
# For the 3rd try we only use bert and multitask but bert_weight < multi_weight and the score doesnt improve.
# Final result: ***submission_b5_d0_m5*** has the best score: ***0.81728***